# GitHub Wrapped — GDS Notebook
## Graph Projection, Louvain Community Detection, PageRank & neo4j-viz

This notebook:
1. Connects to AuraDB using environment-variable credentials
2. Projects a **person-to-person collaboration graph** (persons who co-appear on the same PR as author + reviewer)
3. Runs **Louvain community detection** on the projected graph
4. Writes **community IDs back** to `Person.community` in AuraDB
5. Prints community count and size distribution
6. Runs **PageRank** to identify central reviewers
7. Compares **GDS communities vs GitHub team labels** (majority-label overlap metric)
8. Renders a **neo4j-viz VisualizationGraph** coloured by community
9. Exports **community_assignments.csv** with `login`, `community`, `pagerank_score`

**Prerequisites**
- AuraDB instance with data imported (tasks 001–003 complete)
- GDS plugin enabled on the AuraDB instance (required for `graphdatascience` client)
- Environment variables set: `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`

Install dependencies with:
```bash
uv sync --extra notebook
```

## 1. Setup & Connection

In [ ]:
import os
from dotenv import load_dotenv

# Load credentials from integration.env (searches cwd then parent)
for _p in ['integration.env', '../integration.env']:
    if load_dotenv(_p, override=False):
        print(f'Loaded env from: {_p}')
        break
else:
    print('No integration.env found — using system environment variables')


In [ ]:
from graphdatascience import GraphDataScience

NEO4J_URI      = os.environ['NEO4J_URI']
NEO4J_USERNAME = os.environ['NEO4J_USERNAME']
NEO4J_PASSWORD = os.environ['NEO4J_PASSWORD']
NEO4J_DATABASE = os.environ.get('NEO4J_DATABASE', 'neo4j')

print(f'Connecting to: {NEO4J_URI}')
print(f'Database:      {NEO4J_DATABASE}')

gds = GraphDataScience(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
    database=NEO4J_DATABASE,
)

print(f'GDS version:   {gds.version()}')

## 2. Explore Existing Data

In [ ]:
# Count persons, PRs, AUTHORED and REVIEWED relationships
counts = gds.run_cypher("""
MATCH (p:Person) WITH count(p) AS persons
MATCH (pr:PullRequest) WITH persons, count(pr) AS prs
MATCH ()-[a:AUTHORED]->() WITH persons, prs, count(a) AS authored
MATCH ()-[r:REVIEWED]->() WITH persons, prs, authored, count(r) AS reviewed
RETURN persons, prs, authored, reviewed
""")
print('Data summary:')
print(counts.to_string(index=False))

## 3. Build Collaboration Edge List (Cypher)

The collaboration graph is **person ↔ person** (undirected): two persons are connected if one **authored** and the other **reviewed** the same PR.

We use a Cypher query to compute the co-occurrence count as the edge weight.

In [ ]:
# Preview the top collaboration pairs (author → reviewer, weight = shared PRs)
sample = gds.run_cypher("""
MATCH (author:Person)-[:AUTHORED]->(pr:PullRequest)<-[:REVIEWED]-(reviewer:Person)
WHERE author <> reviewer
WITH author, reviewer, count(DISTINCT pr) AS sharedPRs
ORDER BY sharedPRs DESC
LIMIT 15
RETURN author.login AS author, reviewer.login AS reviewer, sharedPRs
""")
print('Top collaboration pairs (author, reviewer, shared PRs):')
sample

## 4. GDS Graph Projection

We project a **native undirected Person–Person** graph using a Cypher aggregation projection. Each edge weight is the number of PRs where one person authored and the other reviewed.

In [ ]:
GRAPH_NAME = 'collab-graph'

# Drop existing projection if present (makes notebook idempotent)
exists_result = gds.graph.exists(GRAPH_NAME)
if exists_result['exists']:
    gds.graph.drop(GRAPH_NAME)
    print(f'Dropped existing projection: {GRAPH_NAME}')

# Project person-to-person collaboration graph via Cypher aggregation projection.
# The query must end with RETURN gds.graph.project(...).
# Each node is a Person; edge weight = number of shared PRs (author ↔ reviewer on same PR).
g, result = gds.graph.cypher.project(
    """
    MATCH (author:Person)-[:AUTHORED]->(pr:PullRequest)<-[:REVIEWED]-(reviewer:Person)
    WHERE id(author) < id(reviewer)
    WITH author, reviewer, count(DISTINCT pr) AS sharedPRs
    RETURN gds.graph.project($graphName, author, reviewer,
        { relationshipType: 'COLLABORATED',
          relationshipProperties: { weight: sharedPRs }
        },
        { undirectedRelationshipTypes: ['COLLABORATED'] }
    )
    """,
    database=NEO4J_DATABASE,
    graphName=GRAPH_NAME
)

print(f'Graph projected: {GRAPH_NAME}')
print(f'  Nodes:         {g.node_count()}')
print(f'  Relationships: {g.relationship_count()}')
print(f'  Memory:        {result["memoryUsage"]}')

## 5. Louvain Community Detection

In [ ]:
# Run Louvain on the projected graph
# gds.louvain.mutate stores community IDs in the in-memory graph (mutateProperty)
# relationshipWeightProperty='weight' uses the sharedPRs count as edge weight
louvain_result = gds.louvain.mutate(
    g,
    mutateProperty='community',
    relationshipWeightProperty='weight',
    includeIntermediateCommunities=False,
)

print('Louvain result:')
print(f'  Communities found:  {louvain_result["communityCount"]}')
print(f'  Modularity:         {louvain_result["modularity"]:.4f}')
print(f'  Iterations:         {louvain_result["ranIterations"]}')
print(f'  Nodes classified:   {louvain_result["nodePropertiesWritten"]}')

In [ ]:
import pandas as pd

# Stream community assignments from the projected graph to inspect distribution
# gds.louvain.stream re-runs Louvain and returns nodeId + communityId per row
community_df = gds.louvain.stream(
    g,
    relationshipWeightProperty='weight',
)

print(f'Total persons with community: {len(community_df)}')
community_df.head(10)

In [ ]:
# Community size distribution
size_dist = community_df.groupby('communityId').size().reset_index(name='size')
size_dist = size_dist.sort_values('size', ascending=False).reset_index(drop=True)

print(f'Number of communities:   {len(size_dist)}')
print(f'Largest community:       {size_dist["size"].max()} persons')
print(f'Smallest community:      {size_dist["size"].min()} persons')
print(f'Median community size:   {size_dist["size"].median()}')
print()
print('Top 10 largest communities:')
size_dist.head(10)

In [ ]:
# Size distribution histogram (text-based)
bins = [1, 2, 5, 10, 20, 50, 100, 200]
labels = ['1', '2–4', '5–9', '10–19', '20–49', '50–99', '100–199', '200+']
cuts = pd.cut(
    size_dist['size'],
    bins=[0, 1, 4, 9, 19, 49, 99, 199, 10000],
    labels=labels
)
hist = cuts.value_counts().sort_index()
print('Community size distribution:')
for label, cnt in hist.items():
    bar = '█' * min(cnt, 40)
    print(f'  {label:>8} members: {bar} ({cnt})')

## 6. Write Community IDs Back to AuraDB

We write the `community` property from the in-memory projected graph back to the live `Person` nodes in AuraDB.

In [ ]:
# Write community property from projected graph back to AuraDB Person nodes
# gds.louvain.write re-runs Louvain and writes writeProperty to the live database
write_result = gds.louvain.write(
    g,
    writeProperty='community',
    relationshipWeightProperty='weight',
)

print('Community IDs written back to Person nodes in AuraDB.')
print(f'  Nodes updated:     {write_result["nodePropertiesWritten"]}')
print(f'  Communities found: {write_result["communityCount"]}')
print(f'  Modularity:        {write_result["modularity"]:.4f}')

In [ ]:
# Verify: sample Person nodes now have .community set
verify = gds.run_cypher("""
MATCH (p:Person)
WHERE p.community IS NOT NULL
WITH count(p) AS personsWithCommunity
MATCH (p2:Person)
RETURN personsWithCommunity, count(p2) AS totalPersons
""")
print('Verification:')
print(verify.to_string(index=False))

In [ ]:
# Top communities by size (from live AuraDB data)
top_communities = gds.run_cypher("""
MATCH (p:Person)
WHERE p.community IS NOT NULL
WITH p.community AS community, collect(p.login) AS members, count(p) AS size
ORDER BY size DESC
LIMIT 10
RETURN community, size, members[0..5] AS sample_members
""")
print('Top 10 communities by size (from AuraDB):')
top_communities

## 7. PageRank — Identify Central Reviewers

PageRank on the collaboration graph surfaces persons who are structurally central — people who collaborate with many well-connected others. High PageRank typically means a prolific cross-team reviewer.

In [ ]:
# Run PageRank on the in-memory projected graph (mutate stores scores in graph)
pr_result = gds.pageRank.mutate(
    g,
    mutateProperty='pagerank',
    relationshipWeightProperty='weight',
    dampingFactor=0.85,
    maxIterations=20,
)

print('PageRank result:')
print(f'  Iterations ran:   {pr_result["ranIterations"]}')
print(f'  Did converge:     {pr_result["didConverge"]}')
print(f'  Nodes scored:     {pr_result["nodePropertiesWritten"]}')

In [ ]:
# Stream PageRank scores and resolve node IDs to Person logins via AuraDB
pagerank_df = gds.pageRank.stream(
    g,
    relationshipWeightProperty='weight',
    dampingFactor=0.85,
    maxIterations=20,
)

# pagerank_df columns: nodeId, score
# Resolve nodeId → login via a Cypher lookup
node_logins = gds.run_cypher("""
MATCH (p:Person)
RETURN id(p) AS nodeId, p.login AS login
""")
node_logins = node_logins.set_index('nodeId')

pagerank_df = pagerank_df.merge(node_logins, left_on='nodeId', right_index=True, how='left')
pagerank_df = pagerank_df.sort_values('score', ascending=False).reset_index(drop=True)

print('Top 10 central persons by PageRank (collaboration graph):')
top10_pr = pagerank_df[['login', 'score']].head(10)
top10_pr.columns = ['login', 'pagerank_score']
top10_pr = top10_pr.reset_index(drop=True)
top10_pr.index = top10_pr.index + 1  # 1-based rank
print(top10_pr.to_string())

## 8. Community vs Team-Label Comparison

We compare the **GDS-detected communities** against **GitHub team labels** (extracted from the PR `HAS_LABEL` relationship — labels like `team-kernel`, `team-networking`, etc.).

For each community we compute the **majority-label accuracy**: what fraction of community members share the most common team label? A high accuracy indicates the GDS community aligns with a real GitHub team.

> **Metric definition** (majority-label accuracy per community):
> - For each person, extract all team labels from their authored PRs.
> - Each person's "team" = their most-frequent team label.
> - For each community, find the majority team label and count members who share it.
> - Accuracy = `majority_count / community_size`.

In [ ]:
# Fetch each person's most-used team label from their authored PRs.
# Team labels are identified by a 'team-' prefix convention (common in large OSS repos).
# We use APOC-free Cypher: collect all labels per person, sort by count, pick first.
person_teams = gds.run_cypher("""
MATCH (p:Person)-[:AUTHORED]->(pr:PullRequest)-[:HAS_LABEL]->(l:Label)
WHERE l.name STARTS WITH 'team-' OR l.name STARTS WITH 'team/'
WITH p.login AS login, l.name AS teamLabel, count(pr) AS prCount
ORDER BY login, prCount DESC
WITH login, collect(teamLabel)[0] AS primaryTeam
RETURN login, primaryTeam
ORDER BY login
""")

print(f'Persons with a team label: {len(person_teams)}')
print(f'Unique team labels:        {person_teams["primaryTeam"].nunique()}')
print()
print('Team label distribution:')
print(person_teams['primaryTeam'].value_counts().head(15).to_string())

In [ ]:
# Fallback: if no team- labels exist, use the most common label per person (any label)
if len(person_teams) == 0:
    print('No team-prefixed labels found — falling back to top label per person (any label).')
    person_teams = gds.run_cypher("""
    MATCH (p:Person)-[:AUTHORED]->(pr:PullRequest)-[:HAS_LABEL]->(l:Label)
    WITH p.login AS login, l.name AS teamLabel, count(pr) AS prCount
    ORDER BY login, prCount DESC
    WITH login, collect(teamLabel)[0] AS primaryTeam
    RETURN login, primaryTeam
    ORDER BY login
    """)
    print(f'Persons with any label: {len(person_teams)}')
else:
    print('Team labels found — using team-prefixed labels for comparison.')

In [ ]:
# Fetch community assignments from AuraDB (written back in section 6)
community_logins = gds.run_cypher("""
MATCH (p:Person)
WHERE p.community IS NOT NULL
RETURN p.login AS login, p.community AS community
ORDER BY p.login
""")

# Join community assignments with team labels
merged = community_logins.merge(person_teams, on='login', how='left')
merged['primaryTeam'] = merged['primaryTeam'].fillna('<no-label>')

print(f'Persons with community assignment: {len(community_logins)}')
print(f'Persons also with team label:      {(merged["primaryTeam"] != "<no-label>").sum()}')
merged.head(10)

In [ ]:
# Compute majority-label accuracy per community
# Only communities with ≥2 labelled members are meaningful
labelled = merged[merged['primaryTeam'] != '<no-label>'].copy()

overlap_rows = []
for community_id, grp in labelled.groupby('community'):
    community_size_all = (merged['community'] == community_id).sum()
    labelled_size = len(grp)
    if labelled_size < 2:
        continue
    top_label = grp['primaryTeam'].value_counts().idxmax()
    majority_count = (grp['primaryTeam'] == top_label).sum()
    accuracy = majority_count / labelled_size
    overlap_rows.append({
        'community':       community_id,
        'total_size':      community_size_all,
        'labelled_size':   labelled_size,
        'majority_team':   top_label,
        'majority_count':  majority_count,
        'accuracy':        round(accuracy, 3),
    })

overlap_df = pd.DataFrame(overlap_rows).sort_values('accuracy', ascending=False).reset_index(drop=True)

print(f'Communities with ≥2 labelled members: {len(overlap_df)}')
print(f'Mean majority-label accuracy:         {overlap_df["accuracy"].mean():.1%}')
print(f'Median majority-label accuracy:       {overlap_df["accuracy"].median():.1%}')
print()
print('Top communities by majority-label accuracy:')
overlap_df.head(10)

In [ ]:
# Print human-readable summary for the top communities
print('=== Community–Team Alignment Summary ===')
for _, row in overlap_df.head(5).iterrows():
    pct = f'{row["accuracy"]:.0%}'
    print(
        f"  Community {int(row['community'])}: "
        f"{pct} of {row['labelled_size']} labelled members share "
        f"'{row['majority_team']}' (community size: {row['total_size']})"
    )

## 9. neo4j-viz — Community Graph Visualisation

We render the top collaboration pairs as an interactive graph coloured by community using `neo4j-viz`.

In [ ]:
# Fetch top collaboration pairs for visualisation (limit for readability)
# We pull top-50 pairs and resolve community from the merged dataframe
viz_pairs = gds.run_cypher("""
MATCH (author:Person)-[:AUTHORED]->(pr:PullRequest)<-[:REVIEWED]-(reviewer:Person)
WHERE author <> reviewer
  AND author.community IS NOT NULL
  AND reviewer.community IS NOT NULL
WITH author, reviewer, count(DISTINCT pr) AS sharedPRs
ORDER BY sharedPRs DESC
LIMIT 60
RETURN
  author.login    AS authorLogin,
  author.community AS authorCommunity,
  reviewer.login   AS reviewerLogin,
  reviewer.community AS reviewerCommunity,
  sharedPRs
""")

print(f'Visualisation pairs fetched: {len(viz_pairs)}')
viz_pairs.head()

In [ ]:
from neo4j_viz import Node, Relationship, VisualizationGraph

# Assign a stable colour per community (cycle through a palette)
PALETTE = [
    '#58a6ff',  # blue
    '#56d364',  # green
    '#f0883e',  # orange
    '#bc8cff',  # purple
    '#ff7b72',  # red
    '#ffa657',  # amber
    '#39d353',  # bright green
    '#79c0ff',  # light blue
    '#d2a8ff',  # lavender
    '#ff9492',  # salmon
]

# Collect unique persons from the pairs
persons_in_viz: dict[str, dict] = {}
for _, row in viz_pairs.iterrows():
    for login_col, comm_col in [('authorLogin', 'authorCommunity'), ('reviewerLogin', 'reviewerCommunity')]:
        login = row[login_col]
        community = int(row[comm_col]) if pd.notna(row[comm_col]) else -1
        if login not in persons_in_viz:
            persons_in_viz[login] = {'community': community}

# Build stable community → colour mapping from the persons we have
unique_communities = sorted({v['community'] for v in persons_in_viz.values()})
community_colour = {c: PALETTE[i % len(PALETTE)] for i, c in enumerate(unique_communities)}

# Build Node objects
# Node size scales with how many pairs this person appears in (degree in viz subgraph)
person_degree: dict[str, int] = {}
for _, row in viz_pairs.iterrows():
    person_degree[row['authorLogin']]   = person_degree.get(row['authorLogin'], 0) + 1
    person_degree[row['reviewerLogin']] = person_degree.get(row['reviewerLogin'], 0) + 1

max_degree = max(person_degree.values()) if person_degree else 1

nodes = []
for login, info in persons_in_viz.items():
    community = info['community']
    colour = community_colour.get(community, '#888888')
    degree = person_degree.get(login, 1)
    size = 8 + int((degree / max_degree) * 24)  # 8–32 px
    nodes.append(
        Node(
            id=login,
            caption=login,
            color=colour,
            size=size,
        )
    )

# Build Relationship objects
# neo4j-viz Relationship uses source/target (not start_node/end_node)
rels = []
for i, row in viz_pairs.iterrows():
    max_shared = viz_pairs['sharedPRs'].max()
    width = max(1.0, round(row['sharedPRs'] / max_shared * 5, 1))
    rels.append(
        Relationship(
            id=str(i),
            source=row['authorLogin'],
            target=row['reviewerLogin'],
            width=width,
            color='#444c56',
            caption=str(int(row['sharedPRs'])),
        )
    )

print(f'Nodes: {len(nodes)}')
print(f'Edges: {len(rels)}')
print(f'Communities represented: {len(unique_communities)}')

In [ ]:
# Render the collaboration graph coloured by community
vg = VisualizationGraph(nodes=nodes, relationships=rels)
vg.show()

In [ ]:
# Print community colour legend
print('Community colour legend (top communities by size):')
top_comm_ids = overlap_df.head(len(PALETTE))['community'].tolist() if len(overlap_df) > 0 else unique_communities
for comm in unique_communities[:len(PALETTE)]:
    colour = community_colour[comm]
    majority_label = '<unknown>'
    if len(overlap_df) > 0 and comm in overlap_df['community'].values:
        row = overlap_df[overlap_df['community'] == comm].iloc[0]
        majority_label = row['majority_team']
    print(f'  Community {comm:>4}: {colour}  → dominant team: {majority_label}')

## 10. Export Community Assignments to CSV

Export `login`, `community`, and `pagerank_score` to `notebooks/community_assignments.csv`.

In [ ]:
# Build final export DataFrame: login + community + pagerank_score
# pagerank_df was built in section 7; community_logins was built in section 8
export_df = community_logins.merge(
    pagerank_df[['login', 'score']].rename(columns={'score': 'pagerank_score'}),
    on='login',
    how='left',
)
export_df['pagerank_score'] = export_df['pagerank_score'].fillna(0.0).round(6)
export_df = export_df[['login', 'community', 'pagerank_score']].sort_values(
    ['community', 'pagerank_score'], ascending=[True, False]
).reset_index(drop=True)

print(f'Rows to export: {len(export_df)}')
export_df.head(10)

In [ ]:
import pathlib

# Write to notebooks/community_assignments.csv
output_path = pathlib.Path(__file__).parent / 'community_assignments.csv' if '__file__' in dir() else pathlib.Path('community_assignments.csv')
# In Jupyter, __file__ is not defined — use cwd which is typically the notebooks/ dir
output_path = pathlib.Path.cwd() / 'community_assignments.csv'

export_df.to_csv(output_path, index=False)
print(f'Exported {len(export_df)} rows → {output_path}')
print()
print('Columns: login, community, pagerank_score')
print(f'Top PageRank persons:')
print(export_df.nlargest(5, 'pagerank_score')[['login', 'community', 'pagerank_score']].to_string(index=False))

## 11. Cleanup

Drop the in-memory GDS graph projection to free memory. The `Person.community` property persists in AuraDB.

In [ ]:
# Drop the in-memory projection to free server memory
# Person.community values already persisted to AuraDB in the write step above
gds.graph.drop(g)
print(f'Dropped in-memory projection: {GRAPH_NAME}')
print('Person.community values persist in AuraDB.')
print('community_assignments.csv written to notebooks/ directory.')

In [ ]:
# Close the GDS connection
gds.close()
print('Connection closed.')